# 03. 벡터 데이터베이스(VectorDB) 입문 — "의미"로 논문 조각 찾기

02장에서 본 RDB는 숫자·조건 검색에는 완벽했지만, "Cu 배선 저항이 왜 커지나?" 처럼 **뜻으로 찾는 질문** 에는 답하지 못했습니다.
이번 장에서는 문장을 **숫자 벡터** 로 바꾸는 **임베딩(Embedding)** 과, 그 벡터를 저장·검색하는 **벡터 데이터베이스(Vector DB)** 를 배웁니다.
먼저 numpy 로 "가장 나이브한 벡터DB" 를 직접 만든 뒤, 같은 일을 전용 벡터DB인 **Chroma** 로 해 봅니다.

### 학습 목표
- 임베딩이 "의미를 좌표로 바꾸는 것" 임을 재료 물성 좌표 비유로 이해합니다.
- 코사인 유사도(Cosine Similarity)를 numpy 로 직접 계산합니다.
- 실제 임베딩 모델로 **한국어 질문 ↔ 영어 문장** 이 의미로 매칭되는 것을 확인합니다.
- 논문 조각 전체를 임베딩해 top-k 검색 함수를 직접 구현합니다.
- Chroma 로 같은 작업을 하고, 메타데이터 필터·2D 시각화·키워드 검색과의 비교를 통해 벡터DB의 강점과 한계를 이해합니다.

## 환경설정

In [ ]:
# [Colab 전용] 저장소를 내려받고 패키지·Ollama·모델을 설치합니다. (로컬 Jupyter에서는 자동으로 건너뜁니다)
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/skku_rag_tutorials"):
        !git clone -q https://github.com/JDeun/skku_rag_tutorials.git /content/skku_rag_tutorials
    %cd /content/skku_rag_tutorials
    !bash scripts/colab_setup.sh

In [ ]:
import warnings; warnings.filterwarnings("ignore")   # 불필요한 경고 숨기기
import numpy as np                                    # 벡터·행렬 계산
import pandas as pd                                   # 표 출력
import matplotlib.pyplot as plt                       # 그래프
from skku_rag import get_embeddings, load_papers, split_documents, DB_DIR, LLM_MODEL, EMBED_MODEL

print("LLM:", LLM_MODEL, "| 임베딩:", EMBED_MODEL)

## 1. 의미를 숫자로 — 임베딩(Embedding)이란?

재료를 몇 가지 물성으로 표현한다고 해 봅시다. 예를 들어 (밀도, 융점, 비저항) 세 숫자를 정하면 각 재료는 **3차원 공간의 한 점** 이 됩니다.
그러면 **성질이 비슷한 재료는 공간에서 가까이** 모입니다. Cu와 Ag는 가깝고, 세라믹은 멀리 떨어지겠지요.

```
  비저항 ▲
        |            * 세라믹류 (멀리)
        |
        |   * Ru  * Ir
        |  * Rh            ← 성질이 비슷한 금속끼리 가까이
        | * Cu * Al
        +---------------------► 융점
```

**임베딩** 은 이 아이디어를 **문장** 에 적용한 것입니다. 임베딩 모델은 문장을 수백 개의 숫자(벡터)로 바꾸는데,
**뜻이 비슷한 문장일수록 가까운 좌표** 를 갖도록 학습되어 있습니다. 각 숫자가 정확히 무슨 뜻인지는 사람이 해석하기 어렵지만,
"좌표가 가까우면 의미가 비슷하다" 는 성질만 있으면 검색에 쓸 수 있습니다.

### 1-1. 실제 물성 데이터로 '가까움' 보기
02장에서 만든 Sung et al. 2026 Table 1 의 (비저항 ρ0, 평균 자유 행로 λ) 두 값을 2차원 좌표로 써 봅시다.

In [ ]:
# Sung et al. 2026, Table 1 의 값 (ρ0: μΩ·cm, λ: nm)
metals = {"Cu": (1.678, 39.9), "Al": (2.650, 18.9), "Rh": (4.7, 6.88), "Ir": (5.2, 7.09),
          "Ni": (6.93, 5.87), "Mo": (5.34, 11.2), "Nb": (14.75, 2.36)}
names = list(metals)
X = np.array([metals[n] for n in names])   # (7, 2) 행렬: 금속 7개 × 물성 2개

plt.figure(figsize=(5, 3.5))
plt.scatter(X[:, 0], X[:, 1])
for name, (x, y) in zip(names, X):
    plt.annotate(name, (x, y), xytext=(4, 4), textcoords="offset points")
plt.xlabel("bulk resistivity rho0 (uOhm cm)"); plt.ylabel("EMFP lambda (nm)")
plt.title("Metals as 2D points"); plt.show()

In [ ]:
# 두 점 사이 거리(유클리드 거리)로 'Rh와 가장 가까운 금속' 찾기
target = X[names.index("Rh")]
dist = np.linalg.norm(X - target, axis=1)       # 모든 금속과 Rh 사이의 거리
for i in np.argsort(dist):
    print(f"{names[i]:>3}: 거리 {dist[i]:.2f}")

Rh와 가장 가까운 것은 Ir — 실제로도 둘은 비저항·λ 가 비슷한 fcc 귀금속입니다. **"가까움 = 비슷함"** 이 벡터 검색의 전부입니다.

### 1-2. 코사인 유사도(Cosine Similarity)
문장 임베딩에서는 거리 대신 **두 벡터가 가리키는 방향이 얼마나 같은지** 를 주로 씁니다.

$$\text{cos\_sim}(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|}$$

- 1에 가까우면 같은 방향(의미가 비슷), 0이면 무관, −1이면 반대.
- 문장이 길든 짧든(벡터 길이) 상관없이 **내용의 방향** 만 비교하는 장점이 있습니다.

장난감 예시로, 문장마다 세 가지 주제 점수 **(저항/전기, 결정립/미세조직, 요리)** 를 사람이 직접 매겨 3차원 벡터로 만들어 봅니다.

In [ ]:
toy = {  # (저항·전기, 결정립·미세조직, 요리) 주제 점수 — 사람이 손으로 매긴 장난감 벡터
    "Cu resistivity rises in thin lines":  np.array([0.9, 0.3, 0.0]),
    "Grain boundaries scatter electrons":  np.array([0.7, 0.7, 0.0]),
    "Nb addition coarsens Cu grains":      np.array([0.2, 0.9, 0.0]),
    "How to cook kimchi stew":             np.array([0.0, 0.0, 1.0]),
}

def cos_sim(a, b):
    """코사인 유사도: 내적 ÷ (길이 × 길이)"""
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
labels = list(toy)
sim = [[cos_sim(toy[a], toy[b]) for b in labels] for a in labels]
pd.DataFrame(sim, index=labels, columns=[f"S{i+1}" for i in range(len(labels))]).round(2)

In [ ]:
# 처음 두 축(저항, 결정립)만 그려서 '방향' 비교하기
plt.figure(figsize=(4.5, 4))
for i, (label, v) in enumerate(toy.items()):
    plt.arrow(0, 0, v[0], v[1], head_width=0.03, length_includes_head=True, color=f"C{i}")
    plt.text(v[0] + 0.02, v[1] + 0.02, f"S{i+1}", color=f"C{i}")
plt.xlim(-0.05, 1.1); plt.ylim(-0.05, 1.1)
plt.xlabel("topic: resistivity"); plt.ylabel("topic: grain/microstructure")
plt.title("Toy sentence vectors (S4 points out of this plane)"); plt.show()

S1(저항)과 S2(결정립계 산란)는 방향이 비슷해 유사도가 높고, 요리 문장 S4는 다른 축을 향해 유사도가 0입니다.
실제 임베딩 모델은 이런 "주제 축" 을 사람이 정하지 않고 **수백 개 축을 데이터로부터 스스로 학습** 합니다.

## 2. 실제 임베딩 모델로 문장 임베딩하기

In [ ]:
embeddings = get_embeddings()   # 설정된 임베딩 모델 (Ollama)
vec = embeddings.embed_query("Copper resistivity increases as the line width shrinks.")
print("벡터 차원 수:", len(vec))
print("앞의 5개 숫자:", np.round(vec[:5], 4))

문장 하나가 수백 개의 숫자로 바뀌었습니다. 이제 **한국어 문장과 영어 문장** 을 섞어 유사도를 비교해 봅니다.
우리 논문은 영어지만 질문은 한국어로 할 것이므로, **언어가 달라도 뜻이 같으면 가까운지(교차언어 매칭)** 가 중요합니다.

In [ ]:
sentences = [
    "Copper resistivity increases sharply as interconnect dimensions shrink.",  # E1
    "구리 배선이 가늘어지면 비저항이 급격히 커진다.",                              # K1 (E1과 같은 뜻)
    "Ruthenium has a short electron mean free path.",                          # E2
    "루테늄은 전자의 평균 자유 행로가 짧다.",                                     # K2 (E2와 같은 뜻)
    "오늘 점심으로 김치찌개를 먹었다.",                                            # K3 (무관한 문장)
]
tags = ["E1", "K1", "E2", "K2", "K3"]
V = np.array(embeddings.embed_documents(sentences))
V = V / np.linalg.norm(V, axis=1, keepdims=True)   # 길이를 1로 맞추면 내적 = 코사인 유사도
pd.DataFrame(V @ V.T, index=tags, columns=tags).round(3)

**결과 해설**
- E1–K1, E2–K2 처럼 **언어는 달라도 뜻이 같은 쌍** 의 유사도가 대개 높게 나옵니다(교차언어 의미 매칭). 여러분의 표에서 확인해 보세요.
- 무관한 K3(점심 메뉴)는 대부분의 문장과 유사도가 낮게 나오는 것이 보통입니다.
- 다만 절대값(예: 0.6이 높은지)은 모델마다 다릅니다. **"상대적으로 더 높은가"** 로 판단하세요.
- 같은 언어끼리는 주제가 달라도 약간 높게 나오는 경향(언어 자체의 유사성)도 보일 수 있습니다 — 소형 임베딩 모델의 한계입니다.

## 3. 가장 나이브한 벡터DB 직접 만들기

벡터DB의 핵심은 딱 두 가지입니다.
1. **저장**: 모든 문서 조각을 임베딩해 행렬로 쌓아 둔다.
2. **검색**: 질문을 임베딩해 모든 조각과 유사도를 계산하고, 높은 순으로 k개를 고른다(top-k).

XRD 패턴을 데이터베이스의 모든 기준 패턴과 하나하나 비교해 가장 잘 맞는 상(phase)을 찾는 **피크 매칭** 과 똑같습니다.

In [ ]:
# 논문 PDF → 페이지 → 조각(chunk). 01장과 같은 기본값(1000자, 겹침 100자)
from pathlib import Path

chunks = split_documents(load_papers())
PAPER_ID = {"1-s2.0-S1359646216302512-main.pdf": "kim2016",
            "Recent progress in alternative metals for advanced interconnects.pdf": "sung2026"}
for c in chunks:   # 조각마다 '어느 논문인지' 짧은 이름표(메타데이터)를 붙여 둡니다
    c.metadata["paper"] = PAPER_ID[Path(c.metadata["source"]).name]
print("조각 수:", len(chunks), "| 첫 조각 메타데이터:", chunks[0].metadata["paper"], "p.", chunks[0].metadata["page"] + 1)

In [ ]:
import time
start = time.time()
texts = [c.page_content for c in chunks]
matrix = np.array(embeddings.embed_documents(texts))           # (조각 수, 차원 수) 행렬
matrix = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)  # 길이 1로 정규화
print("임베딩 행렬 모양:", matrix.shape, f"| 걸린 시간: {time.time() - start:.1f}초")
np.save(DB_DIR / "naive_vectors_03.npy", matrix)                 # 파일로 저장 = 가장 단순한 '영구 저장'

In [ ]:
def naive_search(query: str, k: int = 3):
    """질문과 가장 비슷한 조각 k개를 (유사도, 조각) 목록으로 돌려줍니다."""
    q = np.array(embeddings.embed_query(query))
    q = q / np.linalg.norm(q)
    scores = matrix @ q                       # 모든 조각과의 코사인 유사도를 한 번에 계산
    top = np.argsort(scores)[::-1][:k]        # 큰 순서로 k개 위치
    return [(float(scores[i]), chunks[i]) for i in top]

def show(results, width=160):
    for score, doc in results:
        text = doc.page_content.replace("\n", " ")[:width]
        print(f"[{score:.3f}] {doc.metadata['paper']} p.{doc.metadata['page'] + 1}: {text}...")

In [ ]:
show(naive_search("구리 배선의 비저항이 선폭이 줄어들면 왜 커지나요?"))

In [ ]:
show(naive_search("Nb를 첨가하면 결정립 크기가 어떻게 변하나요?"))

한국어 질문으로 영어 논문의 관련 문단이 상위에 나왔는지 확인해 보세요. (구체적인 순위와 점수는 임베딩 모델에 따라 달라집니다.) 30줄 남짓한 numpy 코드지만 이것이 벡터 검색의 **본질** 입니다.
(점수 앞 `p.` 는 PDF 쪽 번호입니다.)

## 4. 그런데 왜 전용 벡터DB가 필요할까?

방금 만든 나이브 벡터DB는 조각 183개에서는 충분히 빠릅니다. 하지만 연구실 전체 논문 수만 편이 되면 문제가 생깁니다.

| 문제 | 나이브 numpy 방식 | 전용 벡터DB (Chroma, FAISS, Milvus 등) |
|---|---|---|
| 규모 | 질문마다 **모든** 벡터와 계산 (전수 비교) | **인덱스** 로 후보만 빠르게 비교 |
| 저장 | `.npy` 파일 + 원문/메타데이터는 따로 관리 | 벡터·원문·메타데이터를 한곳에 저장 |
| 추가/삭제 | 행렬을 다시 만들어야 함 | 문서 단위로 추가·삭제 |
| 메타데이터 필터 | 직접 코드 작성 | `filter={"paper": "kim2016"}` 한 줄 |

**근사 최근접 이웃(Approximate Nearest Neighbor, ANN)** 비유: 시료 보관함에서 비슷한 시료를 찾을 때
수만 개를 하나씩 비교하지 않고, 먼저 **"fcc 금속 선반" → "귀금속 칸"** 처럼 범위를 좁힌 뒤 그 안에서만 비교합니다.
가끔 다른 선반에 있는 진짜 1등을 놓칠 수 있지만(그래서 '근사'), 속도는 수십~수천 배 빨라집니다.

## 5. Chroma 로 같은 작업하기

Chroma 는 파이썬에서 쓰기 쉬운 벡터DB입니다. LangChain 의 `Chroma.from_documents` 는 3장에서 우리가 한 일
(**조각 임베딩 → 저장**)을 한 줄로 해 줍니다. 이 장의 DB는 `db/chroma_03` 폴더에 따로 만듭니다(06장의 DB와 겹치지 않게).

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

CHROMA_03 = DB_DIR / "chroma_03"

# 메타데이터는 검색에 쓸 것(논문 이름, 쪽 번호)만 간단히 남깁니다.
docs = [Document(page_content=c.page_content,
                 metadata={"paper": c.metadata["paper"], "page": c.metadata["page"] + 1}) for c in chunks]
# 조각마다 고정된 id → 셀을 다시 실행해도 같은 id 는 덮어써지므로 중복 저장되지 않음
doc_ids = [f"chunk_{i:04d}" for i in range(len(docs))]

In [ ]:
start = time.time()
vectorstore = Chroma.from_documents(
    documents=docs,                     # 저장할 조각들
    ids=doc_ids,                        # 조각 id (다시 실행 시 중복 방지)
    embedding=embeddings,               # 임베딩 모델 (조각을 벡터로 바꿈)
    collection_name="papers_03",        # 컬렉션 이름 (DB 안의 '테이블' 같은 것)
    persist_directory=str(CHROMA_03),   # 디스크에 저장할 폴더
)
print("저장된 조각 수:", vectorstore._collection.count(), f"| {time.time() - start:.1f}초")

In [ ]:
# similarity_search_with_score: (문서, 점수) 목록. Chroma 기본 점수는 '거리' → 작을수록 비슷함!
query = "구리 배선의 비저항이 선폭이 줄어들면 왜 커지나요?"
for doc, dist in vectorstore.similarity_search_with_score(query, k=3):
    print(f"[거리 {dist:.3f}] {doc.metadata['paper']} p.{doc.metadata['page']}: "
          f"{doc.page_content.replace(chr(10), ' ')[:140]}...")

3장의 `naive_search` 와 **같은 조각** 이 나오는지 비교해 보세요. 점수의 뜻만 다릅니다.
나이브 버전은 **유사도(클수록 비슷)**, Chroma 기본값은 **거리(작을수록 비슷)** 입니다.
길이 1로 맞춘 벡터에서는 **(제곱 유클리드) 거리 = 2 − 2 × 코사인 유사도** 관계가 있어서, 예를 들어 유사도 0.48 은 거리 약 1.04 에 해당합니다. 순위는 똑같습니다. 라이브러리는 우리가 만든 것을 편하게 포장한 것일 뿐입니다.

### 5-1. 메타데이터 필터: 특정 논문에서만 찾기
"Kim 2016 논문에서만" 처럼 범위를 좁히고 싶을 때 `filter` 를 씁니다. RDB의 `WHERE` 와 비슷한 역할입니다.

In [ ]:
query = "결정립계 편석이 비저항에 미치는 영향"
for paper in ["kim2016", "sung2026"]:
    print(f"--- filter: paper = {paper}")
    for doc, dist in vectorstore.similarity_search_with_score(query, k=2, filter={"paper": paper}):
        print(f"[거리 {dist:.3f}] p.{doc.metadata['page']}: {doc.page_content.replace(chr(10), ' ')[:120]}...")

In [ ]:
# 저장된 DB 다시 불러오기: 임베딩을 다시 계산하지 않고 폴더에서 바로 읽습니다.
reloaded = Chroma(collection_name="papers_03", embedding_function=embeddings,
                  persist_directory=str(CHROMA_03))
print("다시 불러온 조각 수:", reloaded._collection.count())

## 6. 2D 투영으로 조각 분포 보기 (PCA)

임베딩은 수백 차원이라 눈으로 볼 수 없습니다. **주성분 분석(PCA)** 은 데이터가 가장 넓게 퍼진 두 방향을 찾아
2차원으로 '그림자' 를 비추는 방법입니다. (XRD 다중 패턴을 PCA로 요약해 보는 것과 같은 원리입니다.) numpy 의 SVD 로 직접 계산합니다.

In [ ]:
def pca_2d(M):
    """행렬 M(n×d)을 가장 퍼진 두 방향으로 투영한 n×2 좌표와 투영 함수를 돌려줍니다."""
    mean = M.mean(axis=0)
    _, _, vt = np.linalg.svd(M - mean, full_matrices=False)   # vt[:2] = 주성분 두 방향
    project = lambda A: (A - mean) @ vt[:2].T
    return project(M), project

coords, project = pca_2d(matrix)
papers = np.array([c.metadata["paper"] for c in chunks])
print(coords.shape)

In [ ]:
q = np.array(embeddings.embed_query("구리 배선의 비저항이 선폭이 줄어들면 왜 커지나요?"))
q_xy = project((q / np.linalg.norm(q))[None, :])[0]   # 질문도 같은 2D 평면에 투영

plt.figure(figsize=(6, 4.5))
for pid, color in [("kim2016", "C0"), ("sung2026", "C1")]:
    pts = coords[papers == pid]
    plt.scatter(pts[:, 0], pts[:, 1], s=12, alpha=0.6, color=color, label=f"{pid} ({len(pts)} chunks)")
plt.scatter(*q_xy, marker="*", s=250, color="red", label="query")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend(); plt.title("Chunk embeddings projected to 2D")
plt.show()

**결과 해설**: 대개 같은 논문의 조각들끼리 모여 있는 경향이 보입니다(정확한 모양은 임베딩 모델에 따라 다릅니다). 두 논문의 주제(나노결정 Cu 합금 vs. 배선 대체 금속 리뷰)가 다르기 때문입니다.
빨간 별(질문)이 어느 무리 근처에 있는지도 확인해 보세요. 단, 수백 차원을 2차원으로 눌렀기 때문에
**2D에서 가깝다고 원래 공간에서도 반드시 가까운 것은 아닙니다**(그림자는 정보 일부를 잃습니다).

## 7. 키워드 검색(01장)과 비교

01장의 키워드 검색은 **질문의 단어가 조각에 그대로 등장하는지** 를 셉니다. 간단한 버전을 다시 만들어 비교해 봅니다.

In [ ]:
def keyword_search(query: str, k: int = 3):
    """질문의 단어가 조각에 몇 번 나오는지 세어 많은 순으로 k개를 돌려줍니다."""
    words = [w.lower() for w in query.split()]
    scores = [sum(c.page_content.lower().count(w) for w in words) for c in chunks]
    top = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), chunks[i]) for i in top if scores[i] > 0]

for query in ["구리 비저항 증가 이유", "copper resistivity size effect"]:
    print(f"=== 키워드 검색: {query}")
    show(keyword_search(query), width=100) if keyword_search(query) else print("  결과 없음 (일치하는 단어가 없음)")

In [ ]:
print("=== 벡터 검색: 구리 비저항 증가 이유")
show(naive_search("구리 비저항 증가 이유"), width=100)

| | 키워드 검색 | 벡터 검색 |
|---|---|---|
| 한국어 질문 → 영어 논문 | **실패** (글자가 다름) | 성공 (의미로 매칭) |
| 정확한 용어(예: "PtCoO2", "Cu96M4") | 강함 | 약할 수 있음 (드문 기호는 의미 정보가 적음) |
| 동의어·바꿔 말하기 | 약함 | 강함 |

그래서 실무에서는 둘을 섞는 **하이브리드 검색(Hybrid Search)** 도 많이 씁니다.

## 8. 벡터DB의 한계

벡터 검색은 "비슷한 글" 을 찾을 뿐, **조건을 계산하지는 못합니다.**

In [ ]:
show(naive_search("비저항이 5 μΩ·cm 보다 낮은 금속을 모두 알려줘", k=3), width=120)

**결과 해설**: 비저항 숫자가 많이 등장하는 조각을 가져올 수는 있지만, "5보다 낮은 것만 골라서 **모두**" 라는 조건은 지키지 못합니다.
벡터는 "5" 와 "4.7" 의 크기를 비교할 줄 모르기 때문입니다. 이런 질문은 **02장의 SQL** (`WHERE resistivity_uohm_cm < 5`)이 정확합니다.

또 "Nb 첨가 → 결정립 성장 → 비저항 감소" 처럼 **여러 단계의 관계를 따라가는 질문** 은, 각 단계가 서로 다른 조각에 흩어져 있으면
top-k 몇 개만으로는 연결하기 어렵습니다. 이것이 뒤에서 배울 **그래프 DB / Graph RAG** 의 동기입니다.

| 질문 유형 | 알맞은 도구 |
|---|---|
| 정확한 수치·조건·정렬·집계 | RDB + SQL (02장) |
| 의미·설명·"왜?" | 벡터DB (이번 장) |
| 개체 사이의 관계·다단계 추론 | 그래프DB (뒤의 장) |

## 정리

- **임베딩** 은 문장을 숫자 벡터(기본 모델 기준 수백 차원)로 바꾸며, **뜻이 비슷하면 벡터도 가깝습니다** — 비슷한 물성의 재료가 물성 공간에서 가까운 것과 같습니다.
- **코사인 유사도** = 두 벡터 방향의 일치도. 길이 1로 정규화하면 내적과 같습니다.
- 다국어 임베딩 모델은 **한국어 질문 ↔ 영어 논문** 을 의미로 연결해 줍니다.
- **나이브 벡터DB** = 임베딩 행렬 + 내적 + 정렬. 전용 벡터DB(Chroma)는 여기에 **영구 저장·메타데이터 필터·인덱스(ANN)** 를 더한 것입니다.
- 벡터 검색은 정확한 수치 조건과 관계 추론에 약합니다 → SQL, 그래프DB와 상호 보완.

## 직접 해보기

1. 2장의 문장 목록에 "구리 박막에 Fe를 조금 넣으면 저항이 줄어든다." 와 그 영어 번역을 추가하고, 유사도 표가 어떻게 바뀌는지 확인해 보세요.
2. `naive_search` 에 `paper` 인자를 추가해 특정 논문 조각만 검색하도록 고쳐 보세요. (힌트: `papers == "kim2016"` 인 위치만 점수를 계산하거나, 나머지 점수를 `-1` 로 바꾸기)
3. `split_documents(docs, chunk_size=500, chunk_overlap=50)` 으로 조각을 더 잘게 나눠 같은 질문을 검색해 보세요. 결과의 정확도와 문맥 길이는 어떻게 달라지나요?